In [ ]:
# --- Fast/robust fetch of Stanford Dogs into /content/stanford-dogs (Colab) ---
!sudo apt-get -y -qq install aria2 > /dev/null
!pip install sympy==1.12

In [ ]:
import os, tarfile, subprocess
import torch, json
from pathlib import Path
from torchvision import models


root = "/content/stanford-dogs"
os.makedirs(root, exist_ok=True)
os.chdir(root)

# Canonical URLs (case-sensitive!)
urls = [
  "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar",
  "http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar",  
  "http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar",
]

def have_all_archives():
    return all(os.path.exists(p) for p in ["images.tar", "annotation.tar", "lists.tar"])

def download_with_aria2(urls):
    print("Downloading with aria2c…")
    cmd = ["aria2c", "-x", "8", "-s", "8", "-c"] + urls
    return subprocess.run(cmd).returncode == 0

def download_with_wget(url):
    print(f"wget {url}")
    return subprocess.run(["wget", "-c", url]).returncode == 0

# Try aria2 once; if any file missing, retry missing ones with wget (sequential)
if not have_all_archives():
    download_with_aria2(urls)

for url in urls:
    fname = url.rsplit("/",1)[-1]
    if not os.path.exists(fname):
        download_with_wget(url)

# Final check
missing = [f for f in ["images.tar","annotation.tar","lists.tar"] if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Missing archives after download: {missing}")

print("\nExtracting…")
for t in ["images.tar","annotation.tar","lists.tar"]:
    print("Extracting", t)
    with tarfile.open(t) as tf:
        tf.extractall(path=".")

# Sanity checks
imgs_ok = os.path.isdir(os.path.join(root,"Images"))
ann_ok  = os.path.isdir(os.path.join(root,"Annotation"))
# lists_ok= os.path.isdir(os.path.join(root,"lists"))

print("\nTree:")
print(" - Images exists?", imgs_ok)
print(" - Annotation exists?", ann_ok)
# print(" - lists exists?", lists_ok)

# Extra: verify the .mat files are present
mat_ok = (os.path.exists(os.path.join(root, "train_list.mat")) and
          os.path.exists(os.path.join(root, "test_list.mat")))
print(" - train_list.mat/test_list.mat present?", mat_ok)

print("\nSet CFG['data_root'] =", root)
# ============================================================
# Colab: Stanford Dogs → MobileNetV2 backbone offline training
# ============================================================

# (Optional) Mount Drive if data is there
# from google.colab import drive; drive.mount('/content/drive')

!pip -q install scipy pyyaml

import os, json, math, random, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from scipy.io import loadmat

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchvision import transforms, models

# -------------------------
# Config (edit as you like)
# -------------------------
CFG = {
    # <<< set this to dataset location >>>
    "data_root": "/content/stanford-dogs/",   
    # outputs
    "cleaned_data_path_csv": "/content/dog_breeds_index.csv",
    "cleaned_data_path_json": "/content/dog_breeds_index.json",
    "artifacts_dir": "/content/backbone_artifacts",

    # backbone selection
    "random_seed": 42,
    "num_pretrain_dogs": 10,          # number of backbone classes to pretrain on

    # image + transforms
    "backbone_img_size": 224,

    # training
    "epochs_head_warmup": 3,          # head-only
    "epochs_finetune": 30,            # full fine-tune max
    "patience": 7,                    # early stopping
    "batch_size_train": 64,
    "batch_size_val": 64,
    "lr_head": 1e-3,
    "lr_finetune": 3e-4,
    "backbone_weight_decay": 1e-4,
    "backbone_label_smoothing": 0.05,
}

# -------------------------
# Reproducibility helpers
# -------------------------
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ------------------------------------------------------TEACHER TRAINING PHASE !!!!! 
# GOAL: optimize teacher weights, use strong transforms 
# OUTPUT: Weights of best teacher model

set_seed(CFG["random_seed"])

# -------------------------
# 1) Clean / index dataset
# -------------------------

'''
Writes a clean index:
- dog_breeds_index.csv (rows: split, rel_path, img_path, ann_path, breed, gid)
- dog_breeds_index.json (maps gid → breed)
'''

def _parse_list(mat_path: Path):
    m = loadmat(mat_path)
    files  = [str(x[0]).strip() for x in m['file_list'].squeeze()]
    labels = [int(x) for x in m['labels'].squeeze()]  # 1..120
    return files, labels

def clean_dogs(data_root: str, out_csv: str, out_json: str):
    root = Path(data_root)
    if not (root / "train_list.mat").exists():
        raise FileNotFoundError(
            f"Could not find {root/'train_list.mat'}. "
            "Point CFG['data_root'] to your Stanford Dogs folder."
        )
    tr_files, tr_labels = _parse_list(root/"train_list.mat")
    te_files, te_labels = _parse_list(root/"test_list.mat")

    def rows(files, labels, split):
        for fp, y in zip(files, labels):
            breed = fp.split('/')[0]  # e.g., n02085620-Chihuahua
            yield dict(
                split=split,
                rel_path=fp,
                img_path=str(root/"Images"/fp),
                ann_path=str(root/"Annotation"/(fp.replace('.jpg',''))), # folder + xml name
                breed=breed,
                gid=int(y)-1   # 0..119
            )

    df = pd.DataFrame([*rows(tr_files,tr_labels,"train"), *rows(te_files,te_labels,"test")])
    df.to_csv(out_csv, index=False)

    gid2breed = df.groupby("gid")["breed"].first().sort_index().to_dict()
    Path(out_json).parent.mkdir(parents=True, exist_ok=True)
    Path(out_json).write_text(json.dumps(gid2breed, indent=2))
    print(f"[clean] wrote:\n  {out_csv}\n  {out_json}")
    return df, gid2breed

df_idx, gid2breed = clean_dogs(CFG["data_root"], CFG["cleaned_data_path_csv"], CFG["cleaned_data_path_json"])

# --------------------------------------
# 2) Pick backbone classes (by seed)
# --------------------------------------
'''
	Builds a stable local label map for backbone dog classes : gid_to_local[g] ∈ {0..B-1}
 '''
def pick_backbone_gids(gid2breed: dict, num_pretrain: int, seed: int):
    all_gids = list(map(int, gid2breed.keys()))
    if num_pretrain > len(all_gids):
        raise ValueError(f"num_pretrain_dogs={num_pretrain} > total classes={len(all_gids)}")
    rng = random.Random(seed)
    all_shuf = all_gids[:]
    rng.shuffle(all_shuf)
    chosen = sorted(all_shuf[:num_pretrain])
    print("Backbone dogs:")
    for g in chosen:
        print(f"  gid={g:3d}  breed={gid2breed[g]}")
    return chosen

backbone_gids = pick_backbone_gids(gid2breed, CFG["num_pretrain_dogs"], CFG["random_seed"])

# --------------------------------------
# 3) Datasets / DataLoaders
# --------------------------------------

class DogCsvDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tfm, gid_to_local: dict[int,int]):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.g2l = gid_to_local

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = Image.open(r.img_path).convert("RGB")
        x = self.tfm(img)
        y_local = self.g2l[int(r.gid)]
        return x, y_local

def make_loaders(df_idx: pd.DataFrame, backbone_gids: list[int], img_size: int, bs_tr: int, bs_val: int, device: torch.device):
    # filter to backbone classes
    bb_df = df_idx[df_idx["gid"].isin(backbone_gids)].copy()

    # stable (0..B-1) label map
    local_ids = sorted(backbone_gids)
    gid_to_local = {g:i for i,g in enumerate(local_ids)}

    df_train = bb_df[bb_df["split"]=="train"].copy()
    df_val   = bb_df[bb_df["split"]=="test"].copy()

    mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    ds_tr  = DogCsvDataset(df_train, train_tf, gid_to_local)
    ds_val = DogCsvDataset(df_val,   val_tf,   gid_to_local)

    pin = (device.type == "cuda")
    dl_tr  = DataLoader(ds_tr,  batch_size=bs_tr,  shuffle=True,  num_workers=2, pin_memory=pin)
    dl_val = DataLoader(ds_val, batch_size=bs_val, shuffle=False, num_workers=2, pin_memory=pin)

    return dl_tr, dl_val, gid_to_local, local_ids

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
dl_tr, dl_val, gid_to_local, local_ids = make_loaders(
    df_idx, backbone_gids, CFG["backbone_img_size"],
    CFG["batch_size_train"], CFG["batch_size_val"], device
)

# --------------------------------------
# 4) MobileNetV2 model (ImageNet init)
# --------------------------------------
'''
Initialise the MObileNetV2 Model
'''
def make_mobilenet_v2(num_classes: int, pretrained: bool = True):
    try:
        # torchvision >= 0.13 style
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        net = models.mobilenet_v2(weights=weights)
    except Exception:
        net = models.mobilenet_v2(pretrained=pretrained)
    # replace classifier head
    in_f = net.classifier[-1].in_features  # 1280
    net.classifier[-1] = nn.Linear(in_f, num_classes)
    return net, in_f

num_classes = len(local_ids)
model, feat_dim = make_mobilenet_v2(num_classes=num_classes, pretrained=True)
model.to(device)

# --------------------------------------
# 5) Train loop (head warmup → finetune)
# --------------------------------------
'''
Trains a MobileNetV2 teacher on just the B backbone classes
- ImageNet-initialized MobileNetV2, head replaced with Linear(1280 → B) to fit dog breed dataset.
- Phase 1: head-only warmup → Phase 2: full fine-tuning with early stop.
'''
def per_class_accuracy(model, loader, local_ids, gid2breed, device):
    model.eval()
    per_tot = [0]*len(local_ids)
    per_cor = [0]*len(local_ids)
    tot = 0; cor = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            pred = logits.argmax(1)
            tot += yb.size(0)
            cor += (pred == yb).sum().item()
            for c in range(len(local_ids)):
                m = (yb == c)
                n = int(m.sum().item())
                if n > 0:
                    per_tot[c] += n
                    per_cor[c] += int((pred[m] == yb[m]).sum().item())
    overall = 100.0 * cor / max(1, tot)
    for i, gid in enumerate(local_ids):
        name = gid2breed[gid]
        acc = (100.0 * per_cor[i] / per_tot[i]) if per_tot[i] > 0 else 0.0
        print(f"  {name:35s}: {acc:5.1f}% ({per_cor[i]}/{per_tot[i]})")
    print(f"Overall val acc: {overall:.2f}%")
    return overall

def train_backbone_mnet(model, dl_tr, dl_val, device, cfg):
    label_smoothing = cfg["label_smoothing"]
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # --- Phase 1: head-only warmup ---
    for p in model.features.parameters():
        p.requires_grad = False
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr_head"], weight_decay=cfg["weight_decay"])

    print("\n[Phase 1] Head-only warmup")
    model.train()
    for ep in range(cfg["epochs_head_warmup"]):
        tot, cor, loss_sum = 0, 0, 0.0
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            tot += yb.size(0)
            cor += (logits.argmax(1) == yb).sum().item()
            loss_sum += float(loss.item())
        print(f"  epoch {ep:02d} | train_acc={100*cor/max(1,tot):.1f} | loss={loss_sum/len(dl_tr):.3f}")

    # --- Phase 2: full fine-tune with early stopping ---
    for p in model.features.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr_finetune"], weight_decay=cfg["weight_decay"])

    best_state = None
    best_val = 0.0
    patience = cfg["patience"]
    stall = 0

    print("\n[Phase 2] Full fine-tune")
    for ep in range(cfg["epochs_finetune"]):
        model.train()
        tot, cor, loss_sum = 0, 0, 0.0
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            tot += yb.size(0)
            cor += (logits.argmax(1) == yb).sum().item()
            loss_sum += float(loss.item())
        tr_acc = 100*cor/max(1,tot)
        if ep % 1 == 0:
            print(f"[ep {ep:02d}] train_acc={tr_acc:.1f} | loss={loss_sum/len(dl_tr):.3f}")

        # validation
        model.eval()
        with torch.no_grad():
            v_tot, v_cor = 0, 0
            for xb, yb in dl_val:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                v_tot += yb.size(0)
                v_cor += (logits.argmax(1) == yb).sum().item()
            v_acc = 100*v_cor/max(1,v_tot)
        print(f"[ep {ep:02d}] val_acc={v_acc:.1f}")

        if v_acc > best_val:
            best_val = v_acc
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            stall = 0
            
        else:
            stall += 1

        if stall >= patience:
            print(f"Early stopping at epoch {ep}: best val {best_val:.2f}%")
            break

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)
    return model, best_val

print("\n============================")
print("Training MobileNetV2 backbone")
print("============================")
model, best_val = train_backbone_mnet(model, dl_tr, dl_val, device, CFG)

print("\n--- Per-class validation accuracy (backbone classes) ---")
_ = per_class_accuracy(model, dl_val, local_ids, gid2breed, device)


# ------------------------------------------------------KD CACHING PHASE !!!!! 
# GOAL: freeze teacher, act as a stable labeler 
# steps:
#   1) rebuild the teacher with the saved best checkpoint
#   2) turn off all augmentations, 
#   3) run every image once through the model
#   4) cache clean, reproducible teacher outputs that represent what teacher actually believes about each image
# OUTPUT: teacher logits, features, and ground truth labels

# --------------------------------------
# 6) Save artifacts for later use
# --------------------------------------
'''
Save artifacts meta with the folloiwng:

•	mobilenetv2_backbone.pth (all weights)
•	backbone_meta.json:
•	"backbone": "mobilenet_v2"
•	"feature_dim": 1280 (penultimate width)
•	"num_backbone_classes": B
•	"backbone_gids": [ ... ] (the exact global class IDs in fixed order)
•	"img_size": 224
•	"normalization": mean/std
•	"best_val_acc": <float>

'''
ART = Path(CFG["artifacts_dir"]); ART.mkdir(parents=True, exist_ok=True)
weights_path = ART/"mobilenetv2_backbone.pth"
meta_path    = ART/"backbone_meta.json"

torch.save(model.state_dict(), weights_path)
meta = {
    "backbone": "mobilenet_v2",
    "feature_dim": int(model.classifier[-1].in_features),  # 1280
    "num_backbone_classes": len(local_ids),
    "backbone_gids": local_ids,            # global class IDs used during backbone training
    "img_size": CFG["img_size"],
    "normalization": {"mean":[0.485,0.456,0.406], "std":[0.229,0.224,0.225]},
    "best_val_acc": float(best_val),
}
meta_path.write_text(json.dumps(meta, indent=2))

print(f"\nSaved:")
print(f"  • weights: {weights_path}")
print(f"  • meta   : {meta_path}")


# --------------------------------------
# 7) Reload everything and redefine model to get ready for caching 
# --------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ART = Path("/content/backbone_artifacts")
weights_path = ART/"mobilenetv2_backbone.pth"
meta_path    = ART/"backbone_meta.json"

meta = json.loads(meta_path.read_text())
local_ids = meta["backbone_gids"]
img_size  = meta["img_size"]

# Rebuild same MobileNetV2 head size
teacher = models.mobilenet_v2(weights=None)
in_f = teacher.classifier[-1].in_features
teacher.classifier[-1] = torch.nn.Linear(in_f, len(local_ids))

# Load weights THEN move to device
state = torch.load(weights_path, map_location="cpu")
teacher.load_state_dict(state, strict=True)
teacher.to(device)           # <<< IMPORTANT
teacher.eval()

# ============================================================
# 8) CACHE teacher features+logits (use weak deployment transforms)
# ============================================================

'''
Outputs 2 files, train_cache.pt, val_cache.pt each is a dict with:
	•	"keys": list of image paths (as strings)
	•	"y_local": LongTensor [N] with labels in 0..B-1
	•	"features": FloatTensor [N, 1280] penultimate features
	•	"logits": FloatTensor [N, B] teacher raw logits (no softmax)
	•	"meta": { "backbone_gids": [...], "img_size": 224, "normalization": {...} }
 
 These dicts get used to train student!
'''
print("\nCaching teacher features+logits with deploy transforms...")

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
deploy_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class DogCsvDatasetWithKey(Dataset):
    def __init__(self, df, tfm, gid_to_local):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.g2l = gid_to_local
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = self.tfm(Image.open(r.img_path).convert("RGB"))
        y = self.g2l[int(r.gid)]
        return x, y, r.img_path

bb_df = df_idx[df_idx["gid"].isin(local_ids)].copy()
gid_to_local = {g:i for i,g in enumerate(local_ids)}
df_tr = bb_df[bb_df["split"]=="train"].copy()
df_va = bb_df[bb_df["split"]=="test"].copy()

pin = (device.type == "cuda")
dl_tr_deploy = DataLoader(DogCsvDatasetWithKey(df_tr, deploy_tf, gid_to_local),
                          batch_size=CFG["batch_size_train"], shuffle=False,
                          num_workers=2, pin_memory=pin)
dl_va_deploy = DataLoader(DogCsvDatasetWithKey(df_va, deploy_tf, gid_to_local),
                          batch_size=CFG["batch_size_val"], shuffle=False,
                          num_workers=2, pin_memory=pin)

# single hook + buffer
_buf = {"z": None}
def _hook(module, inp, out):
    _buf["z"] = inp[0].detach()

# hook saves teacher input tensor → the 1280-D penultimate features, allows us to cache logits and features
# logits = output of final layer
# features = input to final laye
h = teacher.classifier[-1].register_forward_hook(_hook)

@torch.no_grad()
def cache_pass(loader):
    feats, logits, ys, keys = [], [], [], []
    for xb, yb, kb in loader:
        xb = xb.to(device, non_blocking=True)   # inputs on same device as teacher
        out = teacher(xb)                        # forward triggers hook
        z = _buf["z"]                            # [N, 1280] on device
        feats.append(z.cpu())
        logits.append(out.cpu())
        ys.append(yb.clone())
        keys.extend(kb)
    return {
        "keys": keys,
        "y_local": torch.cat(ys, 0),
        "features": torch.cat(feats, 0),
        "logits": torch.cat(logits, 0),
        "meta": {"backbone_gids": local_ids, "img_size": img_size,
                 "normalization": {"mean": mean, "std": std}}
    }
# remember logit is raw output after final linear layer, we have Linear(1280 → 10), so for each image a logit is vec len(10)
# remember softmax would turn everything into a probability 
# logits: [N, C] (N = number of samples, C = number of backbone classes)
train_cache = cache_pass(dl_tr_deploy)
val_cache   = cache_pass(dl_va_deploy)
h.remove()

cache_dir = ART / "teacher_cache"
cache_dir.mkdir(parents=True, exist_ok=True)
torch.save(train_cache, cache_dir / "train_cache.pt")
torch.save(val_cache,   cache_dir / "val_cache.pt")
print("Saved teacher caches:")
print(f"  • {cache_dir/'train_cache.pt'}  (features {tuple(train_cache['features'].shape)}, logits {tuple(train_cache['logits'].shape)})")
print(f"  • {cache_dir/'val_cache.pt'}    (features {tuple(val_cache['features'].shape)}, logits {tuple(val_cache['logits'].shape)})")